In [1]:
import os, numpy as np, requests, torch
from types import SimpleNamespace
from dotenv import load_dotenv
from transformers import GPT2TokenizerFast

# context_cite 0.0.4 uses Styler.applymap which was renamed to .map in pandas 2.x
from pandas.io.formats.style import Styler
if not hasattr(Styler, "applymap"):
    Styler.applymap = Styler.map

from context_cite import ContextCiter

load_dotenv(dotenv_path="../.env")
API_KEY  = os.getenv("API_KEY")
BASE_URL = "https://chat.zingplay.com/api/chat/completions"
MODEL    = "local-model-mini"

[nltk_data] Downloading package punkt_tab to /home/coder/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# --- Mock data -----------------------------------------------------------
CONTEXT = (
    "The meal cap for domestic travel is $50 per day. "
    "International travel requires manager approval. "
    "Hotel expenses are reimbursed up to $200 per night. "
    "Flight bookings must be made at least two weeks in advance."
)
QUERY    = "What is the hotel expense reimbursement limit?"
RESPONSE = "The hotel expense reimbursement limit is $200 per night."

In [3]:
# --- Tokenizer wrapper ---------------------------------------------------
#
# ContextCiter and DataCollatorForSeq2Seq need a PreTrainedTokenizerBase, so
# we use GPT-2 (already in the venv) as the shell.  GPT-2 provides:
#   pad / eos tokens, encode/decode, __call__ → BatchEncoding (with
#   token_to_chars), and the full .pad() method DataCollatorForSeq2Seq needs.
#
# We only add one thing: a ChatML chat_template so that apply_chat_template
# produces the same format the zingplay API expects.
#
# The GPT-2 tokenization is used purely for structural bookkeeping inside
# ContextCite (token counts, character spans).  The actual log-probabilities
# come from the API.

tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token   = tokenizer.eos_token
tokenizer.padding_side = "left"

tokenizer.chat_template = (
    "{% for message in messages %}"
    "<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)

In [4]:
# --- API helpers ---------------------------------------------------------

def _actual_tokens(raw_logprobs):
    """
    Return [(decoded_text, logprob)] for the actual input token at each position.

    With prompt_logprobs=1 the API returns, per position:
      1 entry  → that token IS the top-1 prediction (actual = top-1)
      2 entries → actual token was NOT top-1; it is the non-rank-1 entry
    Position 0 is always None (no conditioning); first ChatML token is <|im_start|>.
    """
    result = []
    for entry in raw_logprobs:
        if entry is None:
            result.append(("<|im_start|>", None))
            continue
        if len(entry) == 1:
            v = list(entry.values())[0]
        else:
            # The actual (forced) input token has the highest rank number
            v = max(entry.values(), key=lambda x: x["rank"])
        result.append((v["decoded_token"], v["logprob"]))
    return result


def _extract_user_content(prompt_text: str) -> str:
    """Pull the user-turn text out of a ChatML-formatted string."""
    start = prompt_text.find("<|im_start|>user\n")
    if start == -1:
        raise ValueError(f"No user turn found in prompt: {prompt_text[:80]!r}")
    start += len("<|im_start|>user\n")
    end = prompt_text.find("<|im_end|>", start)
    return prompt_text[start:] if end == -1 else prompt_text[start:end]


def _api_response_token_logprobs(user_content: str, response_text: str):
    """
    Call /chat/completions with prompt_logprobs=1 and return a list of
    (decoded_token, logprob) for every API token that belongs to response_text.

    The model prepends <think>...</think> to the assistant turn, so we locate
    response_text by character matching (rfind) rather than a fixed offset.
    """
    r = requests.post(
        BASE_URL,
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "model": MODEL,
            "messages": [
                {"role": "user",      "content": user_content},
                {"role": "assistant", "content": response_text},
            ],
            "temperature": 0,
            "max_tokens": 1,
            "prompt_logprobs": 1,
        },
    )
    r.raise_for_status()

    tokens   = _actual_tokens(r.json()["prompt_logprobs"])
    full_txt = "".join(t for t, _ in tokens)

    start = full_txt.rfind(response_text)
    if start == -1:
        start = full_txt.rfind(response_text.strip())
    if start == -1:
        raise ValueError(f"response_text not found in decoded sequence.\nTail: {full_txt[-300:]!r}")
    end = start + len(response_text)

    out, pos = [], 0
    for decoded, lp in tokens:
        tok_end = pos + len(decoded)
        if lp is not None and pos >= start and tok_end <= end:
            out.append((decoded, lp))
        pos = tok_end
        if pos >= end:
            break
    return out


def _align_to_gpt2_tokens(api_tokens, response_text: str, gpt2_response_ids: list):
    """
    Map per-API-token logprobs onto per-GPT2-token logprobs using character spans.

    For each GPT-2 token we compute a weighted average of the API logprobs whose
    character spans overlap with it (weighted by overlap length).
    """
    n = len(gpt2_response_ids)
    if not api_tokens:
        return [-5.0] * n

    # Character spans of API tokens (within response_text)
    api_spans, pos = [], 0
    for decoded, lp in api_tokens:
        api_spans.append((pos, pos + len(decoded), lp))
        pos += len(decoded)

    # Character spans of GPT-2 tokens (BatchEncoding.token_to_chars)
    enc = tokenizer(response_text, add_special_tokens=False)

    result = []
    for j in range(n):
        cs = enc.token_to_chars(j)
        our_s, our_e = cs.start, cs.end

        overlapping = [
            (lp, min(our_e, a_e) - max(our_s, a_s))
            for a_s, a_e, lp in api_spans
            if min(our_e, a_e) > max(our_s, a_s)
        ]
        if overlapping:
            total = sum(o for _, o in overlapping)
            result.append(sum(lp * o / total for lp, o in overlapping))
        else:
            mid = (our_s + our_e) / 2
            result.append(min(api_spans, key=lambda x: abs((x[0]+x[1])/2 - mid))[2])
    return result

In [5]:
# --- APIModel ------------------------------------------------------------
#
# Satisfies the interface ContextCiter uses:
#
#   model.device                      → torch.device("cpu")
#   model.generate(input_ids, ...)    → LongTensor [1, full_seq_len]
#   model(input_ids, attention_mask, labels) → obj with .logits [bs, seq, V]
#
# generate():
#   We have a pre-defined RESPONSE so we skip the live API call.
#   We simply return prompt_ids + encode(RESPONSE).  ContextCiter slices off
#   the prompt length in characters to recover the response string.
#
# __call__():
#   1. Decode prompt / response from the batch tensors using GPT-2.
#   2. Parse the ChatML prompt to extract user_content.
#   3. Call the API with user_content + response to get prompt_logprobs.
#   4. Align API token logprobs onto GPT-2 token boundaries.
#   5. Build fake logits so _compute_logit_probs returns the aligned logprobs:
#
#      logits[b, pos, label_id] = api_logprob + log(V-1),  all others = 0
#      → correct_logit - logsumexp(other_logits)
#        = (api_logprob + log(V-1)) - log(V-1)
#        = api_logprob  ✓

class APIModel:

    device = torch.device("cpu")

    def __init__(self, mock_response: str):
        self._mock_response = mock_response

    # ------------------------------------------------------------------
    def generate(self, input_ids: torch.Tensor, **kwargs) -> torch.Tensor:
        resp_ids = tokenizer.encode(self._mock_response, add_special_tokens=False)
        full_ids = input_ids[0].tolist() + resp_ids
        return torch.tensor([full_ids], dtype=torch.long)

    # ------------------------------------------------------------------
    def __call__(self, input_ids, attention_mask=None, labels=None, **kwargs):
        bs, seq_len = input_ids.shape
        V      = tokenizer.vocab_size
        logits = torch.zeros(bs, seq_len, V)
        log_V1 = float(np.log(V - 1))

        for b in range(bs):
            ids = input_ids[b].tolist()
            lbs = labels[b].tolist()

            # Labels are -100 for prompt positions, actual token ids for response
            resp_start   = next(i for i, l in enumerate(lbs) if l != -100)
            prompt_ids   = ids[:resp_start]
            response_ids = lbs[resp_start:]          # actual response token ids

            prompt_text   = tokenizer.decode(prompt_ids)
            response_text = tokenizer.decode(response_ids)
            user_content  = _extract_user_content(prompt_text)

            api_tokens = _api_response_token_logprobs(user_content, response_text)
            aligned    = _align_to_gpt2_tokens(api_tokens, response_text, response_ids)

            for j, (rid, alp) in enumerate(zip(response_ids, aligned)):
                # Position (resp_start - 1 + j) predicts response token j
                pos = resp_start - 1 + j
                logits[b, pos, rid] = alp + log_V1

        return SimpleNamespace(logits=logits)

In [6]:
# --- Wire up ContextCiter ------------------------------------------------
#
# Pass our API-backed wrappers instead of a real HuggingFace model.
# Everything else (partitioning, ablation loop, Lasso regression) runs as-is.

model = APIModel(mock_response=RESPONSE)

cc = ContextCiter(
    model=model,
    tokenizer=tokenizer,
    context=CONTEXT,
    query=QUERY,
    num_ablations=32,
    ablation_keep_prob=0.5,
    batch_size=1,
)

print("Sources:")
for i, s in enumerate(cc.sources):
    print(f"  [{i}] {s}")

Sources:
  [0] The meal cap for domestic travel is $50 per day.
  [1] International travel requires manager approval.
  [2] Hotel expenses are reimbursed up to $200 per night.
  [3] Flight bookings must be made at least two weeks in advance.


In [7]:
# .response triggers generate() — uses mock, no API call
print("Response:", cc.response)

Response: The hotel expense reimbursement limit is $200 per night.


In [8]:
# get_attributions() triggers 32 ablation calls to the API.
# The Lasso solver then fits weights explaining which context sentences
# most affected the model's confidence in the response.
cc.get_attributions(as_dataframe=True, verbose=False)

  0%|          | 0/32 [00:00<?, ?it/s]

/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
/home/coder/dairy_vng/.venv/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


,Score,Source
0,14.411,Hotel expenses are reimbursed up to $200 per night.
1,1.884,The meal cap for domestic travel is $50 per day.
2,-0.177,Flight bookings must be made at least two weeks in advance.
3,-2.560,International travel requires manager approval.
